# Chapter 26 — Regression Deep Dive: OLS, Assumptions, Diagnostics
*Tier 3: All Tracks*

## 🎯 Learning Objectives
- Master OLS inference: t-tests, F-test, R², adjusted R²
- Detect and fix violations: heteroscedasticity, multicollinearity, non-linearity
- Extend to logistic and Poisson regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import pandas as pd
import seaborn as sns

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. OLS — The Full Inference Framework

$$y = X\beta + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2 I)$$

Key statistics:
- $\hat\beta = (X^TX)^{-1}X^Ty$
- $\hat\sigma^2 = \text{RSS}/(n-p-1)$
- $\text{Var}(\hat\beta) = \hat\sigma^2 (X^TX)^{-1}$
- **t-test** for each $\hat\beta_j$: $t = \hat\beta_j / \text{SE}(\hat\beta_j)$
- **F-test** for overall significance: $F = \frac{\text{RSS}_{\text{null}} - \text{RSS}}{p} / \frac{\text{RSS}}{n-p-1}$

In [ ]:
# Boston housing-like data simulation
n = 500
rooms = rng.normal(6, 1.5, n)
crime = rng.exponential(5, n)
dist  = rng.uniform(1, 10, n)
price = 30 + 5*rooms - 0.3*crime - 1.2*dist + rng.normal(0, 5, n)

df = pd.DataFrame({"price": price, "rooms": rooms, "crime": crime, "dist": dist})
X = sm.add_constant(df[["rooms", "crime", "dist"]])
model = sm.OLS(df["price"], X).fit()
print(model.summary())

## 2. Detecting and Fixing Heteroscedasticity

In [ ]:
# Generate heteroscedastic data
x_hetero = rng.uniform(1, 10, 200)
y_hetero = 2*x_hetero + rng.normal(0, x_hetero*0.4, 200)  # variance grows with x

model_ols = sm.OLS(y_hetero, sm.add_constant(x_hetero)).fit()
residuals = model_ols.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(model_ols.fittedvalues, residuals, alpha=0.5, s=20)
axes[0].axhline(0, color="red"); axes[0].set_title("Residuals vs Fitted (Heteroscedastic)")
axes[0].set_xlabel("Fitted"); axes[0].set_ylabel("Residuals")

# Fix: WLS or log-transform
model_log = sm.OLS(np.log(y_hetero), sm.add_constant(np.log(x_hetero))).fit()
res_log = model_log.resid
axes[1].scatter(model_log.fittedvalues, res_log, alpha=0.5, s=20)
axes[1].axhline(0, color="red"); axes[1].set_title("After Log Transform (Fixed)")
axes[1].set_xlabel("Fitted log(y)"); axes[1].set_ylabel("Residuals")
plt.tight_layout(); plt.show()

# Breusch-Pagan test
from statsmodels.stats.diagnostic import het_breuschpagan
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, model_ols.model.exog)
print(f"Breusch-Pagan test: LM={bp_stat:.3f}, p={bp_p:.4f}")
print("Heteroscedasticity detected ✅" if bp_p < 0.05 else "Homoscedastic ✅")

## 3. Logistic and Poisson Regression (GLMs)

In [ ]:
# Logistic regression: probability of default
income = rng.normal(50, 15, 400)
age    = rng.normal(35, 10, 400)
logit  = -2 + 0.03*income - 0.05*age
p_default = 1 / (1 + np.exp(-logit))
default = rng.binomial(1, p_default)

glm_data = pd.DataFrame({"default": default, "income": income, "age": age})
X_glm = sm.add_constant(glm_data[["income", "age"]])
logit_model = sm.Logit(glm_data["default"], X_glm).fit(disp=False)
print(logit_model.summary().tables[1])

# Marginal effects
print("
Average Marginal Effects:")
print(logit_model.get_margeff().summary())

In [ ]:
# Poisson regression: accident counts
exposure = rng.uniform(1, 10, 200)
safety_score = rng.uniform(0, 10, 200)
mu_count = np.exp(0.5 + 0.1*exposure - 0.2*safety_score)
counts = rng.poisson(mu_count)

poisson_data = pd.DataFrame({"count": counts, "exposure": exposure, "safety": safety_score})
X_pois = sm.add_constant(poisson_data[["exposure", "safety"]])
pois_model = sm.GLM(poisson_data["count"], X_pois,
                    family=sm.families.Poisson()).fit()
print(pois_model.summary().tables[1])